# 06. Tree Model Calibration

## Purpose
In this notebook, we refine the "Raw" models from Step 05 to make their probability outputs more accurate.

## Why Calibration?
1.  **The Problem**: Raw tree models (like Random Forest) often output "0.9 ability" when they are actually only 70% sure. They are "overconfident" or "underconfident".
2.  **The Solution**: Calibration maps the raw scores to true probabilities. If the model says 70% risk, it should actually be fraud 70% of the time.
3.  **The Benefit**: This is crucial if we want to show a "Risk Score" to a human analyst.

## Techniques Used:
*   **Sigmoid (Platt Scaling)**: Fits a logistic curve. Good for small data or when the distortion is S-shaped. (Used for XGBoost).
*   **Isotonic Regression**: Fits a non-parametric staircase function. More powerful but can overfit on small data. (Used for others).


### Step 1: Setup and Imports
**What:** Import `CalibratedClassifierCV` from Scikit-Learn.
**Why:** This is the tool that wraps our existing model and learns the correction curve.


In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.calibration import CalibratedClassifierCV
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold

# Add src to path
import sys
sys.path.append('../src')
from utils.data_loader import load_insurance_data
from utils.preprocessing import get_preprocessor

SEED = 42

### Step 2: Load Data & Pre-trained Models
**What:** 
1.  Load the exact same datasets as before.
2.  Load the **Uncalibrated** models we saved in Notebook 05.
**Why:** We are not training from scratch! We are taking the existing "best" models and polishing them.


In [2]:
# Load Data
datasets = {
    'Original': load_insurance_data('preprocessed', verbose=True),
    'Trees': load_insurance_data('trees', verbose=True)
}

# Load Uncalibrated Models
uncalibrated_models = joblib.load('../models/best_tree_models_uncalibrated.joblib')
print("Loaded uncalibrated models.")

✓ Loading preprocessed dataset: insurance_claims_preprocessed_no_hobbies.csv
  Shape: (1000, 51)
  Target distribution: {0: 753, 1: 247}
  Fraud rate: 24.7%
✓ Loading trees dataset: preprocesed_for_trees.csv
  Shape: (1000, 43)
  Target distribution: {0: 753, 1: 247}
  Fraud rate: 24.7%
Loaded uncalibrated models.


In [ ]:

# ============================================================
# step_03 CALIBRATE MODELS
# ============================================================
# Iterate through each model and wrap it in CalibratedClassifierCV

calibrated_models = {}

print("Beginning Calibration...")

for dataset_name, df in datasets.items():
    print(f"\n--- Processing Dataset: {dataset_name} ---")
    X = df.drop('target', axis=1)
    y = df['target']
    
    # Split (Same seed as training)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED
    )
    
    dataset_calibrated = {}
    dataset_uncal = uncalibrated_models.get(dataset_name, {})
    
    if not dataset_uncal:
        print(f"Warning: No valid models found for {dataset_name}")
        continue
        
    for model_name, base_model in dataset_uncal.items():
        print(f"  Calibrating {model_name}...")
        
        # Method selection
        # XGBoost often prefers 'sigmoid' (Platt)
        # Random Forest / Bagging often prefer 'isotonic'
        method = 'sigmoid' if 'XGB' in model_name else 'isotonic'
        
        # Retrain with calibration
        # We must use cross-validation for calibration to avoid data leakage
        model_to_calibrate = clone(base_model)
        
        calib_clf = CalibratedClassifierCV(
            estimator=model_to_calibrate,
            method=method,
            cv=5, 
            n_jobs=-1
        )
        
        calib_clf.fit(X_train, y_train)
        dataset_calibrated[model_name] = calib_clf
        
    calibrated_models[dataset_name] = dataset_calibrated
    
print("\nCalibration Complete.")


### Step 5: Strategic Decision & Saving
**What:** We decide how to use these models in production.
**The Verdict:**
*   **High Recall Mode**: Use the **Uncalibrated** model if we just want to catch *everything* automatedly.
*   **Ranking Mode**: Use the **Calibrated** model if a human is looking at the score.
**Action:** Save BOTH versions to disk.


In [ ]:
# ============================================================
# FINAL CALIBRATION DECISION, SAVING STRATEGY & MODEL SELECTION
# ============================================================

# What we achieved:
# -----------------
# We compared UNCALIBRATED vs CALIBRATED models across two datasets
# ("Original" and "Trees"). Calibration improved PROBABILITY QUALITY 
# (PR-AUC) for several models but also reduced Recall/F2 for others.
#
# Key Insight:
# Calibration is NOT a universal improvement. It is beneficial when:
#   - The model output is used as a probability score (risk ranking UI)
#   - Humans make decisions based on confidence levels
#   - Fairness, transparency, or regulatory demands exist (compliance)
# Calibration may HURT performance when:
#   - The business goal is maximum recall (catch as many fraud cases)
#   - The pipeline uses a fixed F2 threshold only (no probability UI)
#
# Strategic Product Decision:
# --------------------------
# We will save BOTH calibrated and uncalibrated models.
# In production, we want a TOGGLE:
#   MODE: "high_recall"       -> uses uncalibrated (threshold tuned)
#   MODE: "calibrated_ranking"-> uses calibrated (score-based triage)
#
# Model-by-model decision:
# ------------------------
# ExtraTrees (Trees)     -> Calibration IMPROVED scoring stability
# Bagging (Trees)        -> Calibration IMPROVED PR-AUC
# RandomForest (Original)-> Neutral tradeoff; keep both
# XGBoost                -> Calibration REDUCES recall; keep UNCALIBRATED for recall mode
# VotingEnsemble         -> Calibration improves probability quality but reduces recall
#
# Therefore:
# - We SAVE BOTH versions for flexibility.
# - We RECOMMEND calibrated models in scenarios with human triage/priority queues.
# - We USE uncalibrated when capture rate (Recall/F2) is the dominant objective.

# Save uncalibrated and calibrated model sets
calibrated_path = "../models/best_tree_models_calibrated.joblib"
uncalibrated_path = "../models/best_tree_models_uncalibrated.joblib"

joblib.dump(calibrated_models, calibrated_path)
joblib.dump(uncalibrated_models, uncalibrated_path)

print("\n✓ Saved calibrated models to:", calibrated_path)
print("✓ Saved uncalibrated models to:", uncalibrated_path)

# Suggested final production routing (pseudo-spec):
deployment_recommendation = {
    "high_recall_mode": {
        "description": "Aggressive fraud capture (automated flags). False positives acceptable.",
        "use_models": "Uncalibrated",
        "primary_metric": "F2 / Recall",
        "ideal_for": "Automated blocking, auto-verification triggers"
    },
    "calibrated_ranking_mode": {
        "description": "Human-reviewed queue; risk score is shown to an analyst.",
        "use_models": "Calibrated",
        "primary_metric": "PR-AUC / Probability quality",
        "ideal_for": "Triage dashboards, compliance, transparency"
    }
}

print("\n=== DEPLOYMENT MODES AVAILABLE ===")
for mode, cfg in deployment_recommendation.items():
    print(f"\nMODE: {mode}")
    for k,v in cfg.items():
        print(f"  {k}: {v}")


In [4]:
from utils.evaluation import evaluate_model

### Step 4: Verify Improvement (Before vs After)
**What:** Immediately check if our hard work paid off.
**Expectation:** 
*   **PR-AUC (Precision-Recall Area under Curve)** should ideally **increase**, meaning the ranking is better.
*   **Recall** might drop slightly because calibrated probabilities are more conservative (less risky guessing).


In [5]:
# ============================================================
# step_05 CALIBRATION EVALUATION — Compare before/after
# ============================================================
# Why:
# We calibrated models to improve probability quality. Now we verify if 
# calibration actually improved precision/recall curve behavior and PR-AUC.

# Load previously saved models
uncalibrated_models = joblib.load('../models/best_tree_models_uncalibrated.joblib')
# calibrated_models is already loaded above, but re-loading is fine or just use existing
calibrated_models_ref = joblib.load('../models/best_tree_models_calibrated.joblib') 

print("\n=== Comparing UNCALIBRATED vs CALIBRATED Models ===")

for dataset_name, df in datasets.items():
    print(f"\n--- Dataset: {dataset_name} ---")
    X = df.drop('target', axis=1)
    y = df['target']
    
    # Same split logic used during training
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED
    )
    
    uncal = uncalibrated_models.get(dataset_name, {})
    calib = calibrated_models_ref.get(dataset_name, {})
    
    for model_name in uncal.keys():
        print(f"\nModel: {model_name}")
        
        if model_name not in calib:
            print(f"  (No calibrated version found for {model_name})")
            continue
            
        m1 = uncal[model_name]            # Before calibration
        m2 = calib[model_name]            # After calibration
        
        r1 = evaluate_model(m1, X_test, y_test, f"{model_name} UNCALIBRATED")
        r2 = evaluate_model(m2, X_test, y_test, f"{model_name} CALIBRATED")
        
        # Show a compact comparison of key metrics
        print("  Uncalibrated: ", {k: round(r1[k], 4) for k in ['precision','recall','f1','pr_auc','roc_auc']})
        print("  Calibrated:   ", {k: round(r2[k], 4) for k in ['precision','recall','f1','pr_auc','roc_auc']})



=== Comparing UNCALIBRATED vs CALIBRATED Models ===

--- Dataset: Original ---

Model: RandomForest
  Uncalibrated:  {'precision': np.float64(0.6207), 'recall': np.float64(0.7347), 'f1': np.float64(0.6729), 'pr_auc': 0.5356, 'roc_auc': 0.7883}
  Calibrated:    {'precision': np.float64(0.6275), 'recall': np.float64(0.6531), 'f1': np.float64(0.64), 'pr_auc': 0.5639, 'roc_auc': 0.795}

Model: XGBoost
  Uncalibrated:  {'precision': np.float64(0.614), 'recall': np.float64(0.7143), 'f1': np.float64(0.6604), 'pr_auc': 0.5813, 'roc_auc': 0.8079}
  Calibrated:    {'precision': np.float64(0.5897), 'recall': np.float64(0.4694), 'f1': np.float64(0.5227), 'pr_auc': 0.5715, 'roc_auc': 0.804}

Model: ExtraTrees
  Uncalibrated:  {'precision': np.float64(0.6429), 'recall': np.float64(0.7347), 'f1': np.float64(0.6857), 'pr_auc': 0.5614, 'roc_auc': 0.774}
  Calibrated:    {'precision': np.float64(0.6522), 'recall': np.float64(0.6122), 'f1': np.float64(0.6316), 'pr_auc': 0.5855, 'roc_auc': 0.7721}

Model

In [6]:
# ============================================================
# FINAL CALIBRATION DECISION, SAVING STRATEGY & MODEL SELECTION
# ============================================================

# What we achieved:
# -----------------
# We compared UNCALIBRATED vs CALIBRATED models across two datasets
# ("Original" and "Trees"). Calibration improved PROBABILITY QUALITY 
# (PR-AUC) for several models but also reduced Recall/F2 for others.
#
# Key Insight:
# Calibration is NOT a universal improvement. It is beneficial when:
#   - The model output is used as a probability score (risk ranking UI)
#   - Humans make decisions based on confidence levels
#   - Fairness, transparency, or regulatory demands exist (compliance)
# Calibration may HURT performance when:
#   - The business goal is maximum recall (catch as many fraud cases)
#   - The pipeline uses a fixed F2 threshold only (no probability UI)
#
# Strategic Product Decision:
# --------------------------
# We will save BOTH calibrated and uncalibrated models.
# In production, we want a TOGGLE:
#   MODE: "high_recall"       -> uses uncalibrated (threshold tuned)
#   MODE: "calibrated_ranking"-> uses calibrated (score-based triage)
#
# Model-by-model decision:
# ------------------------
# ExtraTrees (Trees)     -> Calibration IMPROVED scoring stability
# Bagging (Trees)        -> Calibration IMPROVED PR-AUC
# RandomForest (Original)-> Neutral tradeoff; keep both
# XGBoost                -> Calibration REDUCES recall; keep UNCALIBRATED for recall mode
# VotingEnsemble         -> Calibration improves probability quality but reduces recall
#
# Therefore:
# - We SAVE BOTH versions for flexibility.
# - We RECOMMEND calibrated models in scenarios with human triage/priority queues.
# - We USE uncalibrated when capture rate (Recall/F2) is the dominant objective.

# Save uncalibrated and calibrated model sets
calibrated_path = "../models/best_tree_models_calibrated.joblib"
uncalibrated_path = "../models/best_tree_models_uncalibrated.joblib"

joblib.dump(calibrated_models, calibrated_path)
joblib.dump(uncalibrated_models, uncalibrated_path)

print("\n✓ Saved calibrated models to:", calibrated_path)
print("✓ Saved uncalibrated models to:", uncalibrated_path)

# Suggested final production routing (pseudo-spec):
deployment_recommendation = {
    "high_recall_mode": {
        "description": "Aggressive fraud capture (automated flags). False positives acceptable.",
        "use_models": "Uncalibrated",
        "primary_metric": "F2 / Recall",
        "ideal_for": "Automated blocking, auto-verification triggers"
    },
    "calibrated_ranking_mode": {
        "description": "Human-reviewed queue; risk score is shown to an analyst.",
        "use_models": "Calibrated",
        "primary_metric": "PR-AUC / Probability quality",
        "ideal_for": "Triage dashboards, compliance, transparency"
    }
}

print("\n=== DEPLOYMENT MODES AVAILABLE ===")
for mode, cfg in deployment_recommendation.items():
    print(f"\nMODE: {mode}")
    for k,v in cfg.items():
        print(f"  {k}: {v}")



✓ Saved calibrated models to: ../models/best_tree_models_calibrated.joblib
✓ Saved uncalibrated models to: ../models/best_tree_models_uncalibrated.joblib

=== DEPLOYMENT MODES AVAILABLE ===

MODE: high_recall_mode
  description: Aggressive fraud capture (automated flags). False positives acceptable.
  use_models: Uncalibrated
  primary_metric: F2 / Recall
  ideal_for: Automated blocking, auto-verification triggers

MODE: calibrated_ranking_mode
  description: Human-reviewed queue; risk score is shown to an analyst.
  use_models: Calibrated
  primary_metric: PR-AUC / Probability quality
  ideal_for: Triage dashboards, compliance, transparency
